# Treino real: grafo de conhecimento sintético (neuro-sym-model)

5 relações, **5000 entidades** (2.5x mais que a rodada anterior), embedding_dim=512, hidden=1024.

**Terceira rodada** -- as duas primeiras (500 e 5000 épocas, só L1) memorizaram sem generalizar. Um sweep de 62 configs em CPU (escala pequena) mostrou que L1 sozinho nunca generaliza, e que o padrão (memoriza vs. colapsa, sem meio-termo) é o mesmo independente de mais épocas/weight_decay -- suspeita: domínio pequeno demais (30 entidades). Ajustes agora: **axiomas de comutatividade ligados** (perda semântica, a outra metade do método DLG que faltava testar), receita de hiperparâmetros do sweep (`weight_decay=0.5`, `gamma_l1=1e-4`, `train_frac=0.4`), e **muito mais entidades** (5000) pra testar se escala do domínio é o fator que faltava. Deve levar ~40-50min na T4.

1. **Ambiente de execução → Alterar tipo de ambiente de execução → GPU (T4)**.
2. Rode a célula de baixo pra confirmar a GPU.
3. Upload do `neurosym_torch_engine.tar.gz` quando pedido.
4. Rode as células seguintes em ordem.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import files
uploaded = files.upload()  # selecione neurosym_torch_engine.tar.gz

In [ ]:
!tar -xzf neurosym_torch_engine.tar.gz
!ls src/neurosym/torch_engine experiments/torch_scale

In [ ]:
import torch
print('torch:', torch.__version__, '| cuda disponivel:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Treino real (5000 entidades, axiomas ligados, 4000 épocas) -- deve levar ~40-50min na T4

In [ ]:
!python experiments/torch_scale/run_pilot.py --device cuda \
    --entities 5000 --relations 5 --arity 3 --facts-per-relation 20000 \
    --embedding-dim 512 --hidden 1024 --epochs 4000 --val-eval-every 25 \
    --train-frac 0.4 --val-frac 0.3 --weight-decay 0.5 --gamma-l1 1e-4

## Baixar o modelo treinado e o resumo (pra trazer de volta pro repositório local)

In [ ]:
from google.colab import files
files.download('experiments/torch_scale/pilot_results/torch_scale_kg_cuda.pt')
files.download('experiments/torch_scale/pilot_results/torch_scale_kg_cuda_summary.json')